In [ ]:
"""
HSV Underwater Object Detection Tuner
exec(open('hsv_tuner.py').read())
"""

import cv2
import numpy as np
import os
import glob
from sklearn.cluster import DBSCAN

# ─────────────────────────────────────────
IMAGE_FOLDER = "./"
IMAGE_EXTS   = ("*.jpg","*.jpeg","*.png","*.bmp","*.tiff")
# ─────────────────────────────────────────

WIN_NAME  = "HSV Tuner"
CTRL_WIN  = "Controls"
GRID_COLS = 3
THUMB_W   = 480
THUMB_H   = 360
PAD       = 6
LABEL_H   = 28

# ── image collection ──────────────────────────────────────────────────
def collect_images(folder):
    paths = []
    for ext in IMAGE_EXTS:
        paths += glob.glob(os.path.join(folder, ext))
        paths += glob.glob(os.path.join(folder, ext.upper()))
    return sorted(set(paths))

# ── white balance (gray world + strength blend) ───────────────────────
def white_balance(img, strength=0.5):
    f  = img.astype(np.float32)
    mb, mg, mr = f[:,:,0].mean(), f[:,:,1].mean(), f[:,:,2].mean()
    k  = (mb + mg + mr) / 3.0
    cor = f.copy()
    cor[:,:,0] = np.clip(f[:,:,0] * k / (mb + 1e-6), 0, 255)
    cor[:,:,1] = np.clip(f[:,:,1] * k / (mg + 1e-6), 0, 255)
    cor[:,:,2] = np.clip(f[:,:,2] * k / (mr + 1e-6), 0, 255)
    return ((1.0 - strength) * f + strength * cor).astype(np.uint8)

# ── morphological merge ───────────────────────────────────────────────
def merge_close(mask, px):
    if px <= 0:
        return mask
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (px*2+1, px*2+1))
    return cv2.erode(cv2.dilate(mask, k), k)

# ── shape filter ──────────────────────────────────────────────────────
def passes_shape(cnt, mode):
    if mode == 0:
        return True
    a = cv2.contourArea(cnt)
    p = cv2.arcLength(cnt, True)
    if a < 1 or p < 1:
        return False
    if mode == 1:
        return (4 * np.pi * a / p**2) > 0.50
    if mode == 2:
        return len(cv2.approxPolyDP(cnt, 0.04*p, True)) == 4
    return True

# ── biggest single contour bbox ───────────────────────────────────────
def biggest_contour_box(contours):
    if not contours:
        return None, None
    c = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(c)
    return c, (x, y, x+w, y+h)

# ── DBSCAN cluster bbox ───────────────────────────────────────────────
def dbscan_box(contours, eps):
    if not contours:
        return None, []
    centres = []
    for c in contours:
        M = cv2.moments(c)
        if M["m00"] == 0: continue
        centres.append([M["m10"]/M["m00"], M["m01"]/M["m00"]])
    if len(centres) < 2:
        x, y, w, h = cv2.boundingRect(contours[0])
        return (x, y, x+w, y+h), [0]*len(contours)
    db     = DBSCAN(eps=eps, min_samples=2).fit(np.array(centres))
    labels = db.labels_
    valid  = set(labels) - {-1}
    if not valid:
        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        return (x, y, x+w, y+h), labels.tolist()
    best = max(valid, key=lambda l: (labels==l).sum())
    idx  = np.where(labels==best)[0]
    pts  = np.vstack([contours[i] for i in idx if i < len(contours)])
    x, y, w, h = cv2.boundingRect(pts)
    return (x, y, x+w, y+h), labels.tolist()

# ── grid / cell helpers ───────────────────────────────────────────────
def cell(img, title):
    c = np.zeros((THUMB_H, THUMB_W, 3), np.uint8)
    c[LABEL_H:] = cv2.resize(img, (THUMB_W, THUMB_H-LABEL_H))
    cv2.rectangle(c, (0,0),(THUMB_W,LABEL_H),(25,25,35),-1)
    cv2.putText(c, title, (8,19), cv2.FONT_HERSHEY_SIMPLEX,
                0.52, (180,220,255), 1, cv2.LINE_AA)
    return c

def make_grid(cells):
    cols  = GRID_COLS
    rows  = int(np.ceil(len(cells) / cols))
    blank = np.zeros((THUMB_H, THUMB_W, 3), np.uint8)
    while len(cells) < rows * cols:
        cells.append(blank)
    return np.vstack([
        np.hstack([np.pad(cells[r*cols+c], ((0,0),(0,PAD),(0,0)),
                          mode="constant") for c in range(cols)])
        for r in range(rows)
    ])

# ── slider read ───────────────────────────────────────────────────────
def S(name):
    return cv2.getTrackbarPos(name, CTRL_WIN)

def nothing(_): pass

PAL = [(255,80,80),(80,255,80),(80,80,255),
       (255,255,80),(255,80,255),(80,255,255)]

# ── MAIN PIPELINE ─────────────────────────────────────────────────────
def process(img):

    # read all sliders
    wb_str  = S("WB Strength") / 100.0
    h_lo    = S("Hue Lo")
    h_hi    = S("Hue Hi")
    s_lo    = S("Sat Lo")
    s_hi    = S("Sat Hi")
    v_lo    = S("Val Lo")
    v_hi    = S("Val Hi")
    min_a   = S("Min Area") * 10
    # max_a   = S("Max Area") * 100
    merge   = S("Merge px")
    shape   = S("Shape 0=Any 1=Circ 2=Rect")
    eps     = max(5, S("DBSCAN eps"))
    mode    = S("Box: 0=Big 1=Clust 2=Both")

    # 1. white balance
    wb = white_balance(img, wb_str)

    # 2. convert to HSV
    hsv = cv2.cvtColor(wb, cv2.COLOR_BGR2HSV)
    h_ch, s_ch, v_ch = cv2.split(hsv)

    # 3. full HSV mask  (hue wraparound for red)
    if h_lo <= h_hi:
        mask = cv2.inRange(hsv,
                           np.array([h_lo, s_lo, v_lo], np.uint8),
                           np.array([h_hi, s_hi, v_hi], np.uint8))
    else:
        m1 = cv2.inRange(hsv,
                         np.array([h_lo, s_lo, v_lo], np.uint8),
                         np.array([180,  s_hi, v_hi], np.uint8))
        m2 = cv2.inRange(hsv,
                         np.array([0,    s_lo, v_lo], np.uint8),
                         np.array([h_hi, s_hi, v_hi], np.uint8))
        mask = cv2.bitwise_or(m1, m2)

    # 4. clean noise
    k    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,5))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)

    # 5. merge close blobs
    merged = merge_close(mask, merge)

    # 6. find + filter contours
    cnts, _ = cv2.findContours(merged, cv2.RETR_EXTERNAL,
                                cv2.CHAIN_APPROX_SIMPLE)
    valid = [c for c in cnts
         if cv2.contourArea(c) >= min_a
         and passes_shape(c, shape)]

    # ── debug vis: individual H S V channels ──
    hue_col = cv2.applyColorMap(
        (h_ch.astype(np.float32)/180*255).astype(np.uint8),
        cv2.COLORMAP_HSV)
    sat_col = cv2.applyColorMap(s_ch, cv2.COLORMAP_BONE)
    val_col = cv2.applyColorMap(v_ch, cv2.COLORMAP_BONE)

    # ── Strategy A: biggest contour ──
    big_cnt, big_box = biggest_contour_box(valid)
    vis_big = wb.copy()
    cv2.drawContours(vis_big, valid, -1, (0,255,100), 1)
    if big_cnt is not None:
        cv2.drawContours(vis_big, [big_cnt], -1, (0,220,255), 2)
    if big_box:
        x1,y1,x2,y2 = big_box
        cv2.rectangle(vis_big,(x1,y1),(x2,y2),(0,200,255),2)
        cv2.putText(vis_big, f"area={int(cv2.contourArea(big_cnt))}",
                    (x1,max(y1-6,14)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,(0,200,255),1,cv2.LINE_AA)

    # ── Strategy B: DBSCAN cluster ──
    clust_box, labels = dbscan_box(valid, eps)
    centres = []
    for c in valid:
        M = cv2.moments(c)
        if M["m00"] == 0: centres.append((0,0)); continue
        centres.append((int(M["m10"]/M["m00"]), int(M["m01"]/M["m00"])))
    vis_clust = wb.copy()
    for i,(cnt,lbl) in enumerate(zip(valid, labels)):
        col = (100,100,100) if lbl==-1 else PAL[lbl%len(PAL)]
        cv2.drawContours(vis_clust,[cnt],-1,col,2)
    for i,(cx,cy) in enumerate(centres):
        lbl = labels[i] if i < len(labels) else -1
        col = (100,100,100) if lbl==-1 else PAL[lbl%len(PAL)]
        cv2.circle(vis_clust,(cx,cy),5,col,-1)
    if clust_box:
        x1,y1,x2,y2 = clust_box
        cv2.rectangle(vis_clust,(x1,y1),(x2,y2),(0,255,255),2)
        cv2.putText(vis_clust,"best cluster",
                    (x1,max(y1-6,14)),cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,(0,255,255),1,cv2.LINE_AA)

    # ── Final box by mode ──
    if   mode == 0: final = big_box
    elif mode == 1: final = clust_box
    else:
        if big_box and clust_box:
            lx1,ly1,lx2,ly2 = big_box
            cx1,cy1,cx2,cy2 = clust_box
            ix1,iy1 = max(lx1,cx1), max(ly1,cy1)
            ix2,iy2 = min(lx2,cx2), min(ly2,cy2)
            final = (ix1,iy1,ix2,iy2) if ix1<ix2 and iy1<iy2 else clust_box
        else:
            final = big_box or clust_box

    result = wb.copy()
    cv2.drawContours(result, valid, -1, (0,255,100), 1)
    if final:
        x1,y1,x2,y2 = final
        cv2.rectangle(result,(x1,y1),(x2,y2),(0,40,255),3)
        cv2.putText(result,
                    f"n={len(valid)}  {x2-x1}x{y2-y1}",
                    (x1,max(y1-8,16)),
                    cv2.FONT_HERSHEY_SIMPLEX,0.55,(0,40,255),2,cv2.LINE_AA)

    # ── HSV overlay debug panel ──
    # shows the three channels side by side in one cell
    hsv_debug = np.hstack([
        cv2.resize(hue_col, (THUMB_W//3, THUMB_H-LABEL_H)),
        cv2.resize(sat_col, (THUMB_W//3, THUMB_H-LABEL_H)),
        cv2.resize(val_col, (THUMB_W//3, THUMB_H-LABEL_H)),
    ])
    hsv_cell = np.zeros((THUMB_H, THUMB_W, 3), np.uint8)
    hsv_cell[LABEL_H:] = hsv_debug
    cv2.rectangle(hsv_cell,(0,0),(THUMB_W,LABEL_H),(25,25,35),-1)
    cv2.putText(hsv_cell,"3  H | S | V channels",
                (8,19),cv2.FONT_HERSHEY_SIMPLEX,0.52,(180,220,255),1,cv2.LINE_AA)
    # add small labels inside
    for i,lbl in enumerate(["H","S","V"]):
        cv2.putText(hsv_cell, lbl,
                    (i*(THUMB_W//3)+6, LABEL_H+18),
                    cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),1,cv2.LINE_AA)

    return make_grid([
        cell(img,                                        "1  Original"),
        cell(wb,                                         "2  White Balance"),
        hsv_cell,
        cell(cv2.cvtColor(mask,   cv2.COLOR_GRAY2BGR),  "4  HSV Mask"),
        cell(cv2.cvtColor(merged, cv2.COLOR_GRAY2BGR),  "5  Merged Mask"),
        cell(vis_big,                                    "6  Biggest Contour"),
        cell(vis_clust,                                  "7  DBSCAN Clusters"),
        cell(result,                                     "8  FINAL"),
    ])

# ── SETUP ─────────────────────────────────────────────────────────────
images = collect_images(IMAGE_FOLDER)
if not images:
    raise FileNotFoundError(f"No images in '{IMAGE_FOLDER}'")
print(f"{len(images)} images  |  [ prev   ] next   q quit")

idx = 0

# guard against Jupyter re-runs creating duplicate sliders
try:
    cv2.getTrackbarPos("Hue Lo", CTRL_WIN)
    print("Reusing existing Controls window.")
except:
    cv2.namedWindow(CTRL_WIN, cv2.WINDOW_NORMAL)
    cv2.resizeWindow(CTRL_WIN, 520, 480)
    for name, default, maxv in [
        ("WB Strength",                50,  100),  # 0=original 100=full WB
        ("Hue Lo",                      0,  180),
        ("Hue Hi",                    180,  180),
        ("Sat Lo",                    100,  255),  # key for false positive reduction
        ("Sat Hi",                    255,  255),
        ("Val Lo",                     50,  255),
        ("Val Hi",                    255,  255),
        ("Min Area",                    1,  500),  # x10 px²
        # ("Max Area",                  500,  500),  # x100 px²
        ("Merge px",                   15,   80),
        ("Shape 0=Any 1=Circ 2=Rect",   0,    2),
        ("DBSCAN eps",                 60,  300),
        ("Box: 0=Big 1=Clust 2=Both",   0,    2),
    ]:
        cv2.createTrackbar(name, CTRL_WIN, default, maxv, nothing)

cv2.namedWindow(WIN_NAME, cv2.WINDOW_NORMAL)

while True:
    img = cv2.imread(images[idx])
    if img is None:
        idx = (idx+1) % len(images)
        continue

    out = process(img)
    cv2.putText(out,
                f"[{idx+1}/{len(images)}] {os.path.basename(images[idx])}",
                (12, out.shape[0]-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (160,160,160), 1, cv2.LINE_AA)
    cv2.imshow(WIN_NAME, out)

    key = cv2.waitKey(30) & 0xFF
    if   key in (ord('q'), 27): break
    elif key == ord(']'): idx = (idx+1) % len(images)
    elif key == ord('['): idx = (idx-1) % len(images)

cv2.destroyAllWindows()